# पाठ 10 - उत्पादन में एआई एजेंट्स

इस पाठ में आप Microsoft Agent Framework (Python) का उपयोग करके एआई एजेंट्स के **उत्पादन पैटर्न** सीखेंगे। हम निम्नलिखित विषयों को कवर करते हैं:

- **निरीक्षणीयता** — एजेंट इंटरैक्शन में समय निर्धारण और लॉगिंग जोड़ना
- **मूल्यांकन** — प्रतिक्रिया गुणवत्ता को स्कोर करने के लिए एक मूल्यांकन एजेंट का उपयोग
- **लागत प्रबंधन** — टोकन अनुकूलन और मॉडल चयन के लिए रणनीतियाँ

परिदृश्य एक **यात्रा एजेंट** है जो उपयोगकर्ताओं को यात्राएँ योजना बनाने में मदद करता है, जिसके ऊपर निगरानी और मूल्यांकन की परतें होती हैं।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचार-विमर्श

AI एजेंट्स को प्रोटोटाइप से प्रोडक्शन में ले जाने के लिए तीन मुख्य स्तंभों पर सावधानीपूर्वक ध्यान देने की आवश्यकता होती है:

1. **पर्यवेक्ष्यता (Observability)** — आपको यह देखना होगा कि एजेंट क्या कर रहा है, इसमें कितना समय लगता है, और यह कौन से टूल्स कॉल करता है। ट्रेसिंग और लॉगिंग के बिना, प्रोडक्शन समस्याओं का डिबग करना लगभग असंभव होता है।

2. **मूल्यांकन (Evaluation)** — स्वचालित गुणवत्ता जांच यह सुनिश्चित करती है कि एजेंट के उत्तर समय के साथ सटीक, पूर्ण और सहायक बने रहें। एक मूल्यांकन एजेंट परिभाषित मानदंडों के खिलाफ उत्तरों को स्कोर कर सकता है।

3. **लागत प्रबंधन (Cost Management)** — टोकन उपयोग सीधे लागत को प्रभावित करता है। प्रॉम्प्ट ऑप्टिमाइज़ेशन, मॉडल चयन, और कैशिंग जैसी रणनीतियाँ गुणवत्ता को प्रभावित किए बिना खर्चों को नियंत्रण में रखने में मदद करती हैं।


## एक अवलोकनीय एजेंट बनाना

हम यात्रा उपकरणों को परिभाषित करते हैं और लेटेंसी पर नज़र रखने के लिए एजेंट कॉल को टाइमिंग के साथ रैप करते हैं। उत्पादन में आप OpenTelemetry या इसी तरह के ट्रेसिंग बैकएंड के साथ एकीकरण करेंगे।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन पैटर्न

एक सामान्य उत्पादन पैटर्न है कि दूसरे एजेंट का उपयोग **मूल्यांकनकर्ता** के रूप में किया जाता है। मूल्यांकनकर्ता प्राथमिक एजेंट के उत्तर को पूर्वनिर्धारित मानदंडों जैसे पूर्णता, सटीकता, और सहायकता के खिलाफ स्कोर करता है।

यह सक्षम बनाता है:
- उपयोगकर्ताओं तक उत्तर पहुँचने से पहले स्वचालित गुणवत्ता गेट्स
- संकेत या मॉडल बदलने पर प्रतिगमन पहचान
- समय के साथ एजेंट प्रदर्शन की निरंतर निगरानी


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत प्रबंधन रणनीतियाँ

उत्पादन AI एजेंट्स के लिए लागत नियंत्रण महत्वपूर्ण है। यहाँ प्रमुख रणनीतियाँ दी गई हैं:

| रणनीति | विवरण |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | सिस्टम निर्देशों को संक्षिप्त रखें। इनपुट टोकन कम करने के लिए अनावश्यक संदर्भ निकालें। |
    "| **मॉडल चयन** | सरल कार्यों जैसे वर्गीकरण या निष्कर्षण के लिए छोटे, सस्ते मॉडल (जैसे GPT-4.1-mini) का उपयोग करें, और जटिल तर्क के लिए बड़े मॉडलों को आरक्षित रखें। |\n",
| **कैशिंग** | पुनरावर्ती API कॉल से बचने के लिए उपकरण परिणामों और बार-बार पूछे जाने वाले प्रश्नों को कैश करें। |
| **टोकन बजट** | अप्रत्याशित रूप से लंबे उत्तरों को रोकने के लिए `max_tokens` सीमाएँ निर्धारित करें। |
| **बैचिंग** | जहाँ संभव हो, कई उपयोगकर्ता प्रश्नों को एकल API कॉल में समूहित करें। |

व्यवहार में, एक स्तरीय दृष्टिकोण अच्छा काम करता है: सीधे अनुरोधों को एक तेज़, सस्ते मॉडल पर भेजें और केवल जटिल प्रश्नों को अधिक सक्षम मॉडल को बढ़ाएं।


## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. **एजेंट इंटरैक्शन** में टाइमिंग और लॉगिंग के साथ प्रेक्षणीयता जोड़ें, जो ट्रेसिंग और मॉनिटरिंग के लिए आधार तैयार करता है।
2. **एजेंट प्रतिक्रियाओं का मूल्यांकन करें** एक मूल्यांकन एजेंट का उपयोग करके जो पूर्णता, सटीकता और सहायकता को स्कोर करता है।
3. **लागत प्रबंधन** करें प्रॉम्प्ट ऑप्टिमाइजेशन, मॉडल चयन, कैशिंग और टोकन बजट के माध्यम से।

ये उत्पादन पैटर्न सुनिश्चित करते हैं कि आपके AI एजेंट विश्वसनीय, मापन योग्य और पैमाने पर लागत-प्रभावी हैं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
